# Stage 3A - Direct LLM Benchmark (Improved)
## Thinking model (`deepseek-ai/DeepSeek-R1-Distill-Qwen-14B`) | 5-class benchmark

**Baseline problems diagnosed:**

| Metric | Baseline | Root cause |
|---|---|---|
| Valid_Prediction_Rate | 65.4% | `max_tokens=128` truncates reasoning — no label output |
| Strict_Format_Rate | 45.9% | Model gets cut off mid-chain |
| Accuracy | 55.6% | ~35% samples have no valid prediction |

**Improvements applied:**

| Issue | Fix |
|---|---|
| `max_tokens=128` cuts reasoning chain | Raised to **512** |
| No examples of expected format | Added **3-shot examples** |
| No constraint on reasoning length | Prompt asks for brief reasoning |

This notebook ran on Colab with GPU A100.

Saved outputs: per-sample predictions CSV, summary JSON, one-row metrics CSV, confusion matrix PNG.

In [ ]:
!pip install -q vllm pandas scikit-learn matplotlib seaborn tqdm

from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

In [ ]:
import os
import re
import json
import time
import math
import numpy as np
import pandas as pd
from tqdm import tqdm

from vllm import LLM, SamplingParams
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    mean_squared_error,
    mean_absolute_error,
)
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ============================================
# Paths and basic configuration
# ============================================
DATA_PATH  = "/content/drive/MyDrive/Yelp_Project/data/processed/test_data.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Yelp_Project/LLM/deepseek_r1_distill_qwen14b_5class_improved"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TEXT_COL  = "text"
LABEL_COL = "stars"

TASK_TYPE  = "5_class"
RUN_ID     = "deepseek_r1_distill_qwen14b_5class_improved"
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"

BATCH_SIZE     = 16
MAX_MODEL_LEN  = 4096
MAX_NEW_TOKENS = 512   # ← was 128; reasoning model needs room to finish CoT

LABELS         = [1, 2, 3, 4, 5]
DISPLAY_LABELS = ["1", "2", "3", "4", "5"]

In [ ]:
# ============================================
# Load and prepare the test set
# ============================================
df_raw = pd.read_csv(DATA_PATH)
df_raw = df_raw[[TEXT_COL, LABEL_COL]].copy()
df_raw[LABEL_COL] = df_raw[LABEL_COL].astype(int)

df_test = df_raw.copy()
df_test["original_stars"] = df_test[LABEL_COL]
df_test["label"] = df_test[LABEL_COL]
EVAL_LABEL_COL = "label"

print(f"Loaded {len(df_test)} samples for {TASK_TYPE}")
print(df_test[[TEXT_COL, "original_stars", "label"]].head())

In [ ]:
# ============================================
# Improved prompt with 5-shot examples (one per star level)
# ============================================

FEW_SHOT_EXAMPLES = """Here are five examples, one for each star rating:

Review: "Absolutely horrible. Raw chicken, 90-minute wait, and the waiter was dismissive. Never coming back."
Reasoning: Severe food safety issue, extremely long wait, rude staff. Worst possible experience.
Final label: 1

Review: "The food was okay but nothing special. Service was slow and the place was noisy. Probably won't return."
Reasoning: Mediocre food, slow service, uncomfortable environment. Below average but not catastrophic.
Final label: 2

Review: "Decent spot. Food was fine, service was adequate, nothing remarkable but no complaints either."
Reasoning: Everything is average — food, service, ambiance. Solidly middle-of-the-road experience.
Final label: 3

Review: "Really enjoyed dinner here. Great flavors, attentive staff, nice atmosphere. Minor issue with parking."
Reasoning: Positive food and service experience with only a trivial inconvenience. Good but not perfect.
Final label: 4

Review: "Phenomenal! Every dish was exquisite, the staff went above and beyond, and the ambiance was magical."
Reasoning: Superlatives across all dimensions — food, service, atmosphere. Flawless experience.
Final label: 5
"""

def build_prompt(text: str) -> str:
    return (
        "You are a Yelp review star-rating predictor.\n"
        "Task: predict the star rating from 1 to 5 based on the review text.\n\n"
        + FEW_SHOT_EXAMPLES +
        "Now classify the following review.\n"
        "Reason briefly (2-3 sentences), then output EXACTLY this as your LAST line:\n"
        "Final label: X\n"
        "where X is an integer from 1 to 5. Do not add anything after that line.\n\n"
        f"Review: {text}"
    )

def parse_output(text: str):
    text = str(text).strip()
    # Strict: last 'Final label: X' (1-5)
    matches = re.findall(r"Final label:\s*([1-5])", text, flags=re.IGNORECASE)
    if matches:
        return int(matches[-1]), True
    # Fallback: last standalone digit 1-5
    star_matches = re.findall(r"\b([1-5])\b", text)
    if star_matches:
        return int(star_matches[-1]), False
    return None, False

# Sanity check
sample = build_prompt("Great tacos!")
print(sample[:700])

In [ ]:
# ============================================
# Model setup
# ============================================
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,
)

llm = LLM(
    model=MODEL_NAME,
    trust_remote_code=True,
    max_model_len=MAX_MODEL_LEN,
)

In [ ]:
# ============================================
# Batched inference
# ============================================
all_rows = []
start_time = time.time()

for start in tqdm(range(0, len(df_test), BATCH_SIZE), desc="Inference"):
    end = min(start + BATCH_SIZE, len(df_test))
    batch_df = df_test.iloc[start:end].copy()

    prompts = [build_prompt(text) for text in batch_df[TEXT_COL].tolist()]
    outputs = llm.generate(prompts, sampling_params)

    for i, output in enumerate(outputs):
        raw_text = output.outputs[0].text
        pred, strict_ok = parse_output(raw_text)

        all_rows.append({
            "text"            : batch_df.iloc[i][TEXT_COL],
            "original_stars"  : int(batch_df.iloc[i]["original_stars"]),
            "true_label"      : int(batch_df.iloc[i][EVAL_LABEL_COL]),
            "raw_output"      : raw_text,
            "pred"            : pred,
            "strict_format_ok": strict_ok,
        })

eval_time_seconds = time.time() - start_time

result_df = pd.DataFrame(all_rows)
pred_path = os.path.join(OUTPUT_DIR, f"{RUN_ID}_predictions.csv")
result_df.to_csv(pred_path, index=False, encoding="utf-8")

print(f"Saved predictions to: {pred_path}")
print(f"Eval time (s): {eval_time_seconds:.2f}")
print(result_df.head())

In [ ]:
# ============================================
# Evaluation
# ============================================
eval_df = result_df.dropna(subset=["pred"]).copy()
eval_df["pred"] = eval_df["pred"].astype(int)

y_true = eval_df["true_label"].to_numpy()
y_pred = eval_df["pred"].to_numpy()

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
cm       = confusion_matrix(y_true, y_pred, labels=LABELS)
mse      = mean_squared_error(y_true, y_pred)
mae      = mean_absolute_error(y_true, y_pred)

valid_prediction_rate = result_df["pred"].notna().mean()
strict_format_rate    = result_df["strict_format_ok"].mean()
eval_speed            = len(result_df) / eval_time_seconds if eval_time_seconds > 0 else np.nan

off_by_1_acc = np.mean(np.abs(y_pred - y_true) <= 1)

incorrect_mask = (y_pred != y_true)
if incorrect_mask.sum() > 0:
    adj_error_ratio = np.mean(np.abs(y_pred[incorrect_mask] - y_true[incorrect_mask]) == 1)
else:
    adj_error_ratio = 0.0

mid_confusion_ratio = np.mean(
    np.isin(y_true, [2, 3, 4]) &
    np.isin(y_pred, [2, 3, 4]) &
    (y_true != y_pred)
)

report = classification_report(
    y_true, y_pred, labels=LABELS, output_dict=True, digits=4
)

summary = {
    "Run_ID"               : RUN_ID,
    "Task_Type"            : TASK_TYPE,
    "Mode"                 : "Prompt_Inference_Improved",
    "Model"                : MODEL_NAME,
    "Batch_Size"           : BATCH_SIZE,
    "Max_New_Tokens"       : MAX_NEW_TOKENS,
    "Few_Shot"             : True,
    "Accuracy"             : float(accuracy),
    "Macro_F1"             : float(macro_f1),
    "Valid_Prediction_Rate": float(valid_prediction_rate),
    "Strict_Format_Rate"   : float(strict_format_rate),
    "Off_By_1_Acc"         : float(off_by_1_acc),
    "Adj_Error_Ratio"      : float(adj_error_ratio),
    "Mid_Confusion_Ratio"  : float(mid_confusion_ratio),
    "MSE"                  : float(mse),
    "MAE"                  : float(mae),
    "Eval_Time(s)"         : float(eval_time_seconds),
    "Eval_Speed(s/s)"      : float(eval_speed),
    "Confusion_Matrix"     : cm.tolist(),
    "Classification_Report": report,
}

summary_path = os.path.join(OUTPUT_DIR, f"{RUN_ID}_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

metrics_row = pd.DataFrame([{
    "Run_ID"               : RUN_ID,
    "Task_Type"            : TASK_TYPE,
    "Mode"                 : "Prompt_Inference_Improved",
    "Model"                : MODEL_NAME,
    "Batch_Size"           : BATCH_SIZE,
    "Max_New_Tokens"       : MAX_NEW_TOKENS,
    "Few_Shot"             : True,
    "Accuracy"             : accuracy,
    "Macro_F1"             : macro_f1,
    "Valid_Prediction_Rate": valid_prediction_rate,
    "Strict_Format_Rate"   : strict_format_rate,
    "Off_By_1_Acc"         : off_by_1_acc,
    "Adj_Error_Ratio"      : adj_error_ratio,
    "Mid_Confusion_Ratio"  : mid_confusion_ratio,
    "MSE"                  : mse,
    "MAE"                  : mae,
    "Eval_Time(s)"         : eval_time_seconds,
    "Eval_Speed(s/s)"      : eval_speed,
}])

metrics_csv_path = os.path.join(OUTPUT_DIR, f"{RUN_ID}_metrics.csv")
metrics_row.to_csv(metrics_csv_path, index=False, encoding="utf-8")

print("Saved summary JSON to:", summary_path)
print("Saved metrics CSV to :", metrics_csv_path)
metrics_row

In [ ]:
# ============================================
# Confusion matrix
# ============================================
plt.figure(figsize=(8, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=DISPLAY_LABELS,
    yticklabels=DISPLAY_LABELS,
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title(f"Confusion Matrix - {RUN_ID}")
plt.tight_layout()

fig_path = os.path.join(OUTPUT_DIR, f"{RUN_ID}_confusion_matrix.png")
plt.savefig(fig_path, dpi=200)
plt.show()
print("Saved confusion matrix to:", fig_path)

In [ ]:
# ============================================
# Comparison with baseline run
# ============================================
baseline = {
    "Run_ID"               : "5class_fulltest (baseline)",
    "Max_New_Tokens"       : 128,
    "Few_Shot"             : False,
    "Accuracy"             : 0.5560,
    "Macro_F1"             : 0.5448,
    "Valid_Prediction_Rate": 0.6545,
    "Strict_Format_Rate"   : 0.4589,
    "Off_By_1_Acc"         : 0.8993,
}
improved = {
    "Run_ID"               : f"{RUN_ID} (improved)",
    "Max_New_Tokens"       : MAX_NEW_TOKENS,
    "Few_Shot"             : True,
    "Accuracy"             : accuracy,
    "Macro_F1"             : macro_f1,
    "Valid_Prediction_Rate": valid_prediction_rate,
    "Strict_Format_Rate"   : strict_format_rate,
    "Off_By_1_Acc"         : off_by_1_acc,
}

cmp_df = pd.DataFrame([baseline, improved]).set_index("Run_ID")
print("\n=== Baseline vs Improved ===")
print(cmp_df.to_string())

In [ ]:
# ============================================
# Final console summary
# ============================================
print(f"Run_ID               : {RUN_ID}")
print(f"Task_Type            : {TASK_TYPE}")
print(f"Mode                 : Prompt_Inference_Improved")
print(f"Model                : {MODEL_NAME}")
print(f"Max_New_Tokens       : {MAX_NEW_TOKENS}")
print(f"Few_Shot             : True")
print(f"Accuracy             : {accuracy:.4f}")
print(f"Macro_F1             : {macro_f1:.4f}")
print(f"Valid_Prediction_Rate: {valid_prediction_rate:.4f}")
print(f"Strict_Format_Rate   : {strict_format_rate:.4f}")
print(f"Off_By_1_Acc         : {off_by_1_acc:.4f}")
print(f"Adj_Error_Ratio      : {adj_error_ratio:.4f}")
print(f"Mid_Confusion_Ratio  : {mid_confusion_ratio:.4f}")
print(f"MSE                  : {mse:.6f}")
print(f"MAE                  : {mae:.6f}")
print(f"Eval_Time(s)         : {eval_time_seconds:.2f}")
print(f"Eval_Speed(s/s)      : {eval_speed:.4f}")